# The gravitational train inside a relativistic star — a computational exercise

*Supplementary material to* I. Rosignoli, **“The Gravitational Train in the Interior Schwarzschild Metric:
NLO and NNLO Perturbative Expansions”**, submitted to *European Journal of Physics* (2026).

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ImanetorX98/CoordinateTimeGravityTrain/blob/main/notebooks/gravity_train_exercise.ipynb)

**What you will do.** A particle is released from rest at one point of the surface of a homogeneous star and
slides without friction through a tunnel to another surface point. You will compute the tunnel that minimises
the coordinate travel time by integrating the exact relativistic Snell law, and you will compare it with the
Newtonian hypocycloid and with the NLO and NNLO formulas of the paper.

**Prerequisites.** Sections II–VI of the paper. Basic Python (functions, arrays, plots).

**How to use this notebook.** Run the cells in order with **Shift + Enter**. Grey cells contain code; you may
change any number (for example the compactness `mu`) and run the cell again. Each part includes a
**check value**: if you obtain it, you are on the right track. Difficulty: ★ basic, ★★ intermediate,
★★★ advanced.

| Part | Content | Level |
|---|---|---|
| 1 | Metric functions and optical index | ★ |
| 2 | Newtonian limit | ★ |
| 3 | Two numerical pitfalls and how to remove them | ★★ |
| 4 | Exact relativistic tunnels: the exact values of Table I | ★★ |
| 5 | Newton, NLO, NNLO and exact compared | ★★ |
| 6 | Open explorations: proper time; loss of optimality of the diameter | ★★★ |

Units: $R = c = 1$, $x = r/R$, compactness $\mu = r_s/R$ with $0 < \mu < 8/9$.

In [ ]:
import numpy as np
from scipy.integrate import quad
from scipy.optimize import brentq
import matplotlib.pyplot as plt

PLASMA = plt.get_cmap("plasma")   # same palette as the figures of the paper
NEWTON = "#1f77b4"

## Part 1 — Metric functions and optical index ★

The interior Schwarzschild metric (paper, Eqs. 8–10) has
$$A(x) = \tfrac14\left[3\sqrt{1-\mu} - \sqrt{1-\mu x^2}\right]^2, \qquad B(x) = \frac{1}{1-\mu x^2}.$$

For a particle released from rest at the surface, the coordinate travel time is $t = \int n\, d\ell$ with
the effective optical index (Eq. 26)
$$n(x) = \frac{1}{\sqrt{A(x)}\,\sqrt{1 - A(x)/A(1)}}.$$

Here $d\ell^2=B(x)\,dx^2+x^2d\varphi^2$ is spatial proper length, rather than Euclidean length.
The coordinate $t$ is normalised to a static clock at infinity; surface clocks measure $\sqrt{A(1)}t$.
The tunnel constrains the particle, with a normal force doing no work; it is the tunnel's optical path,
rather than the particle's spacetime trajectory, that is a geodesic.

*Why this matters.* $n$ plays the role of a refractive index: a stationary tunnel has the same variational equations as a ray
in an effective medium with index $n$ on the spatial metric. This is an analogy for a massive particle. It combines two effects: $1/\sqrt{A}$ (clocks deep in the star run slow with respect to infinity) and $1/\sqrt{1-A/A(1)}$ (the inverse of the local speed fixed by energy conservation).

It is convenient to work with $w(x) = 1/n(x)^2$, which is finite everywhere:

In [ ]:
def A(x, mu):
    return 0.25 * (3 * np.sqrt(1 - mu) - np.sqrt(1 - mu * x**2))**2

def B(x, mu):
    return 1 / (1 - mu * x**2)

def w_coord(x, mu):
    # 1/n^2 for the coordinate-time problem, Eq. (26)
    return A(x, mu) * (1 - A(x, mu) / A(1, mu))

x = np.linspace(0, 0.995, 400)
fig, ax = plt.subplots(figsize=(5, 3.2))
for mu, col in zip([0.05, 0.3, 0.6, 0.85], [0.15, 0.4, 0.62, 0.82]):
    n = 1 / np.sqrt(w_coord(x, mu))
    ax.plot(x, n / n[0], color=PLASMA(col), label=fr"$\mu={mu}$")
ax.set_yscale("log"); ax.set_xlabel("$x = r/R$"); ax.set_ylabel("$n(x)/n(0)$")
ax.legend(frameon=False); plt.show()

**Question 1.1.** All curves diverge at $x \to 1$. Why? *Hint:* where is the particle at rest?
Is the same true in Newtonian gravity (paper, Eq. 3)?

**Question 1.2.** For $\mu = 0.85$ the index *decreases* away from the centre much more steeply than for small
$\mu$. Which of the two factors in $n$ is responsible?

**Check value.** `1/np.sqrt(w_coord(0, 0.3))` should print about `3.07`.

In [ ]:
print(1 / np.sqrt(w_coord(0, 0.3)))

## Part 2 — The Newtonian limit ★

The Snell law $n(x)\,x\,\sin\alpha = C$ with $C = n(q)\,q$ (Eqs. 31–32) gives the half opening angle of a
tunnel whose deepest point is at $x = q$ (Eq. 34):
$$\frac{\Delta}{2} = \int_q^1 \frac{C\sqrt{B(x)}}{x\sqrt{n^2(x)\,x^2 - C^2}}\,dx
= \int_q^1 \frac{q\,\sqrt{B(x)}\,\sqrt{w(x)/w(q)}}{x\sqrt{x^2 - q^2\,w(x)/w(q)}}\,dx .$$

In Newtonian gravity $n \propto (1-x^2)^{-1/2}$, i.e. $w = 1 - x^2$, $B = 1$, and the exact answer is
$q = 1 - \Delta/\pi$ (Eq. 7). Let us first integrate the formula *as it is written*, with SciPy's `quad`:

In [ ]:
def w_newton(x, mu=0.0):
    return 1 - x**2

def B_newton(x, mu=0.0):
    return 1.0

def integrand(x, q, w, Bf, mu):
    r = w(x, mu) / w(q, mu)
    return q * np.sqrt(Bf(x, mu) * r) / (x * np.sqrt(x**2 - q**2 * r))

q = 0.5
value, err = quad(integrand, q, 1, args=(q, w_newton, B_newton, 0.0))
print("Delta/2 =", value, "  exact:", np.pi / 2 * (1 - q))

**Check value.** For $q = 0.5$ you should get $\Delta/2 \approx 0.7854 = \pi/4$, i.e. $\Delta = 90^\circ$.

`quad` gets the right answer here, but it has to work hard, because the integrand is infinite at $x = q$.
Part 3 shows why this becomes a real problem.

*Why this matters.* An analytic limit gives an independent test before using GR.
**Question 2.1.** Why does multiplying $n$ by a constant leave the tunnel shape unchanged?

## Part 3 — Two numerical pitfalls and how to remove them ★★

1. **Turning point.** At $x = q$ the denominator vanishes like $\sqrt{x - q}$: the tunnel is horizontal there.
2. **Surface.** At $x = 1$ the particle starts at rest, so $n \to \infty$, $w \to 0$, and the angular
   integrand tends to zero: harmless, as long as the formula is written with $w$ rather than $n$.

The turning-point singularity is *integrable*, but it slows a simple quadrature rule down. Try the midpoint rule
on an even grid:

In [ ]:
def midpoint_naive(q, w, Bf, mu, N):
    # midpoint rule: never evaluates the integrand at the endpoints, where the direct formula is singular
    h = (1 - q) / N
    x = q + h * (np.arange(N) + 0.5)
    return h * np.sum(integrand(x, q, w, Bf, mu))

exact = np.pi / 4
for N in [10, 100, 1000, 10000]:
    print(f"N = {N:6d}   error = {abs(midpoint_naive(0.5, w_newton, B_newton, 0.0, N) - exact):.1e}")

The error decreases only like $N^{-1/2}$. The cure is the substitution used for the Newtonian hypocycloid
(Eq. 4):
$$x^2 = q^2 + (1 - q^2)\sin^2 u, \qquad 0 \le u \le \pi/2 .$$
Then $\sqrt{x^2 - q^2} = \sqrt{1-q^2}\,\sin u$ and $dx = (1-q^2)\sin u \cos u\,du / x$: the factor $\sin u$
cancels the singularity at the turning point, and $\cos u$ tames the surface.

In [ ]:
def angle_smooth(u, q, w, Bf, mu):
    # integrand of Eq. (34) after the substitution x^2 = q^2 + (1 - q^2) sin^2 u
    x = np.sqrt(q**2 + (1 - q**2) * np.sin(u)**2)
    return integrand(x, q, w, Bf, mu) * (1 - q**2) * np.sin(u) * np.cos(u) / x

def half_angle(q, w, Bf, mu):
    # Delta/2 for a tunnel with turning radius q
    return quad(angle_smooth, 0, np.pi / 2, args=(q, w, Bf, mu), limit=200)[0]

def midpoint_smooth(q, w, Bf, mu, N):
    h = np.pi / 2 / N
    u = h * (np.arange(N) + 0.5)
    return h * np.sum(angle_smooth(u, q, w, Bf, mu))

for N in [10, 100, 1000]:
    print(f"N = {N:5d}   error = {abs(midpoint_smooth(0.5, w_newton, B_newton, 0.0, N) - exact):.1e}")

**Check value.** With the substitution, $N = 10$ gives an error of about $10^{-9}$, and $N = 100$ gives an
error below $10^{-13}$.

**Question 3.1.** Why does a *change of variable* improve a numerical integral so dramatically, although the
integral itself is unchanged?

## Part 4 — Exact relativistic tunnels: the exact values of Table I ★★

For a given separation $\Delta$ we need the turning radius $q$ such that `half_angle(q) = Δ/2`.
For $\mu \lesssim 0.75$ the opening angle decreases steadily as $q$ grows, so there is exactly one
solution; the bracket used below works for separations away from $0^\circ$ and $180^\circ$. Above $\mu_s \simeq 0.769$ (Eq. 73) this is
no longer true near the diameter: the Snell law can have several solutions, and only a comparison of
travel times tells which one is fastest (Part 6b). The function below therefore prints a warning there.

In [ ]:
def find_q(Delta, w, Bf, mu):
    # turning radius q of the tunnel with opening Delta (0 < Delta < pi)
    if mu > 0.75:
        print(f"Warning: mu = {mu} > 0.75; the Snell law may have several solutions "
              "and this one need not be the fastest (see Part 6b).")
    # q very close to 0 or 1 is avoided: there the ratio w(x)/w(q) loses digits
    return brentq(lambda q: half_angle(q, w, Bf, mu) - Delta / 2, 1e-4, 0.999, xtol=1e-12)

Delta = np.pi / 2
print("Newtonian:", find_q(Delta, w_newton, B_newton, 0.0))
for mu in [0.05, 0.1, 0.2, 0.3, 0.4]:
    print(f"mu = {mu:4.2f}   q_exact = {find_q(Delta, w_coord, B, mu):.5f}")

**Check values (Table I of the paper).** For $\Delta = 90^\circ$: 0.50803, 0.51662, 0.53584, 0.55851, 0.58587.

**Question 4.1.** $q$ grows with $\mu$: the optimal tunnel becomes *shallower*. Explain the competition between faster local motion at depth and the
coordinate-time cost from redshift, using Part 1. The spatial factor $B$ also enters the optimum.

## Part 5 — Newton, NLO, NNLO and exact compared ★★

The perturbative result of the paper (Eq. 59), with $q_0 = 1 - \Delta/\pi$, is
$$q = q_0 + \mu\,\frac{q_0(1-q_0^2)(15-7q_0^2)}{32}
+ \mu^2\,\frac{q_0(1-q_0^2)(1827 - 3489q_0^2 + 2457q_0^4 - 539q_0^6)}{4096} + O(\mu^3).$$

In [ ]:
def q_series(Delta, mu):
    q0 = 1 - Delta / np.pi
    q1 = q0 * (1 - q0**2) * (15 - 7 * q0**2) / 32
    q2 = q0 * (1 - q0**2) * (1827 - 3489 * q0**2 + 2457 * q0**4 - 539 * q0**6) / 4096
    return q0, q0 + mu * q1, q0 + mu * q1 + mu**2 * q2

print(" mu     q_Newton  q_NLO    q_NNLO   q_exact   err_NLO   err_NNLO")
for mu in [0.05, 0.1, 0.2, 0.3, 0.4, 0.5]:
    q0, qn, qnn = q_series(Delta, mu)
    qe = find_q(Delta, w_coord, B, mu)
    print(f"{mu:4.2f}   {q0:.5f}  {qn:.5f}  {qnn:.5f}  {qe:.5f}   {abs(qn-qe)/qe:.1e}   {abs(qnn-qe)/qe:.1e}")

Now draw the tunnels. The shape follows by accumulating the angle $\varphi$ along the tunnel:

In [ ]:
def tunnel(q, w, Bf, mu, n=300):
    # (X, Y) of the whole tunnel; the two stations are symmetric about the vertical axis
    us = np.linspace(0, np.pi / 2, n)
    f = lambda u: angle_smooth(u, q, w, Bf, mu)
    phi = np.concatenate([[0], np.cumsum([quad(f, a, b)[0] for a, b in zip(us[:-1], us[1:])])])
    assert abs(phi[-1] - half_angle(q, w, Bf, mu)) < 1e-8   # check the accumulated angle
    x = np.sqrt(q**2 + (1 - q**2) * np.sin(us)**2)
    r = np.concatenate([x[::-1], x[1:]]); p = np.concatenate([-phi[::-1], phi[1:]])
    return r * np.sin(p), r * np.cos(p)

fig, ax = plt.subplots(figsize=(5.5, 3))
th = np.linspace(-np.pi / 2, np.pi / 2, 300); ax.plot(np.sin(th), np.cos(th), "k", lw=1)
ax.plot(*tunnel(1 - Delta / np.pi, w_newton, B_newton, 0.0), color=NEWTON, lw=2, label="Newton")
for mu, col in zip([0.1, 0.3, 0.5, 0.7], [0.2, 0.42, 0.62, 0.82]):
    ax.plot(*tunnel(find_q(Delta, w_coord, B, mu), w_coord, B, mu), color=PLASMA(col), lw=2,
            label=fr"exact, $\mu={mu}$")
ax.set_aspect("equal"); ax.legend(fontsize=7, frameon=False, ncol=2)
plt.show()

**Check value.** The relative error of NNLO is about $1.7\times10^{-4}$ at $\mu = 0.1$ and
$5\times10^{-3}$ at $\mu = 0.3$ (paper, Sec. VI F).

**Question 5.1 — what determines the range of validity of a perturbative expansion?**
The NNLO error grows roughly like $\mu^3$ at small $\mu$. Check this from your table. Why should this low-order expansion not be trusted arbitrarily close to
$\mu = 8/9$? *Hint:* look at what happens to $A(0)$ (paper, Eq. 65).

**Exercise 5.2.** Repeat the table for $\Delta = 30^\circ$ and $\Delta = 150^\circ$. For which
separations is the expansion more accurate?

## Part 6 — Open explorations ★★★

### 6a. Coordinate time or proper time?

The stations measure the coordinate time (up to the constant factor $\sqrt{A(1)}$ of their own clocks).
A passenger's watch measures the proper time $\tau$. From Eq. (22) of the paper,
$d\tau = d\ell \big/ \sqrt{A(1)/A - 1}$, so minimising $\tau$ is again a Fermat problem, with a different
index $n_\tau$ (same release from rest at the surface).

**Exercise.** Find $w_\tau = 1/n_\tau^2$, write it in the cell below, and use `find_q` to compute the
proper-time optimum for $\Delta = 90^\circ$ and $\mu = 0.3$. Is the passenger's fastest tunnel deeper or
shallower than the stations' one? Explain the difference physically.

In [ ]:
def w_proper(x, mu):
    # Exercise: replace the next line with your expression for 1/n_tau^2
    raise NotImplementedError("write w_proper first")

# When w_proper is ready, remove the '#' in front of the next two lines and run the cell again:
# print("q_tau =", find_q(np.pi / 2, w_proper, B, 0.3))
# print("q_t   =", find_q(np.pi / 2, w_coord, B, 0.3))

<details><summary><b>Solution (click to open)</b></summary>

$w_\tau(x) = A(1)/A(x) - 1$, i.e. `return A(1, mu) / A(x, mu) - 1`.

For $\mu = 0.3$, $\Delta = 90^\circ$ this gives $q_\tau \approx 0.527$, against $q_t \approx 0.559$
for the stations' optimum: both are shallower than the Newtonian tunnel ($q = 0.5$), but the passenger's
fastest tunnel goes *deeper*. For a fixed surface-release energy, the two indices differ by the factor $n_t/n_\tau = \sqrt{A(1)}/A(x)$, which
grows towards the centre: seen from the stations, every second spent deep in the star costs extra
coordinate time, because clocks there run slow; the passenger's own watch runs slow too, so for the
passenger the deep, fast part of the trip is not penalised as much.
</details>

### 6b. Is the straight diameter always the fastest antipodal tunnel?

For $\Delta = 180^\circ$ the diameter is always a geodesic. In Newtonian gravity it is also the fastest
tunnel. The paper (Sec. VII B) shows that this stops being true above $\mu_s \simeq 0.769$.

**Exercise.** For several values of $\mu$ between 0.7 and 0.85, compute the opening angle of a *nearly*
radial tunnel, `2 * half_angle(0.01, w_coord, B, mu)`, in degrees. At which $\mu$ does it exceed
$180^\circ$? What does that mean for the rays leaving one station close to the diameter? *Hint:* think of a
converging lens. A conjugate point is where neighbouring stationary rays from one station refocus.
Above the threshold the diameter is stationary but ceases to be a local minimum.

In [ ]:
for mu in [0.70, 0.74, 0.76, 0.77, 0.78, 0.80, 0.85]:
    print(f"mu = {mu:4.2f}   opening of a near-radial tunnel = "
          f"{np.degrees(2 * half_angle(0.01, w_coord, B, mu)):8.3f} deg")

**Check value.** The opening crosses $180^\circ$ between $\mu = 0.76$ and $\mu = 0.77$.

**Going further.** Above that threshold a pair of mirror-image curved tunnels is faster than the diameter.
To verify it you need the travel time,
$t/2 = \int_q^1 \sqrt{B}\;x \,/\, \big(\sqrt{w}\,\sqrt{x^2 - q^2 w/w(q)}\big)\,dx$; compare the diameter
and the curved tunnel at $\mu = 0.8$ (paper: $t = 14.230$ against $13.886$, in units $R/c$).
Here $w=1/n^2$; the paper's alternative $W=A[A(1)-A]$ obeys $W=A(1)w$.
The diameter ($q=0$) is integrated separately; the curved antipodal tunnel is found by searching
$q$ between 0.1 and 0.8, away from the diameter.

**Question 6b.1.** Why is solving the Snell equation alone insufficient to identify the fastest tunnel here?

In [ ]:
def travel_time(q, mu):
    if q == 0:
        # x=sin(u) removes the surface singularity for the diameter.
        def f(u):
            x = np.sin(u)
            return np.sqrt(B(x, mu)/w_coord(x, mu)) * np.cos(u)
    else:
        # same substitution as in half_angle
        def f(u):
            x = np.sqrt(q**2 + (1 - q**2) * np.sin(u)**2)
            r = w_coord(x, mu) / w_coord(q, mu)
            return (np.sqrt(B(x, mu) / w_coord(x, mu)) * (1 - q**2) * np.sin(u) * np.cos(u)
                    / np.sqrt(x**2 - q**2 * r))
    return 2 * quad(f, 0, np.pi/2, epsabs=2e-9, epsrel=2e-9)[0]

mu = 0.8
qc = brentq(lambda q: 2*half_angle(q, w_coord, B, mu)-np.pi, 0.1, 0.8)
print(f"Curved antipodal branch: q = {qc:.6f}")
print(f"Diameter: {travel_time(0, mu):.3f} R/c; curved: {travel_time(qc, mu):.3f} R/c")